# Stage 1: Select Implied Copyable Trades

Find profitable follower BUYs that follow a leader's BUY or SELL on the same
token within a time window. Two separate leader groups: **buy leaders** (whose
BUYs precede follower BUYs) and **sell leaders** (whose SELLs precede follower
BUYs).

Grid-search over selection thresholds to maximize **copyable PnL from implied
trades** on the validation split.

**Output:** `stage1_implied_result.json` with best selection params.

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from lib import (
    load_trades,
    split_data,
    compute_copyable_notional,
    compute_opening_metrics,
    select_follower_wallets,
    select_leader_wallets,
    detect_implied_buys,
    score_leaders,
    evaluate_implied_pnl,
    evaluate_follower_buy_performance,
    iterative_leader_follower_filter,
    filter_stable_leaders,
    filter_leaders_by_drawdown,
    filter_followers_by_drawdown,
    filter_followers_by_val_roi,
    filter_pairs_by_frequency,
    run_implied_grid_search,
    DEFAULT_TAGS
)
from polymarket_analysis.wallet_selection.volatility import compute_wallet_metrics


pd.options.display.float_format = "{:.4f}".format
pd.options.display.max_rows = 100

print(f"Tags: {DEFAULT_TAGS}")

Tags: {'Weather'}


## Parameters

In [2]:
if DEFAULT_TAGS == {"Weather"}:
    PARAMS = dict(
        # --- Follower selection ---
        min_follower_copyable_roi=0.30,
        min_follower_trade_value=100,
        min_follower_num_buckets=30,
        max_follower_hhi=0.3,
        # --- Leader selection ---
        min_leader_trade_count=20,
        max_leader_hhi=1,
        # --- Broad follower set for leader stability detection ---
        stability_min_follower_roi=0.0,
        stability_min_follower_buckets=10,
        stability_max_follower_hhi=1,
        # --- Leader filtering (drawdown-based) ---
        max_dd_pnl_ratio=0.3,
        # --- Follower filtering (drawdown-based) ---
        follower_max_dd_pnl_ratio=0.3,
        # --- Pair filtering ---
        min_pair_observations=3,
        # --- Time window ---
        time_window_minutes=15,
    )
elif DEFAULT_TAGS == {"Politics"}:
    PARAMS = dict(
        # --- Follower selection ---
        min_follower_copyable_roi=0.30,
        min_follower_trade_value=100,
        min_follower_num_buckets=30,
        max_follower_hhi=0.3,
        # --- Leader selection ---
        min_leader_trade_count=20,
        max_leader_hhi=1,
        # --- Broad follower set for leader stability detection ---
        stability_min_follower_roi=0.0,
        stability_min_follower_buckets=10,
        stability_max_follower_hhi=1,
        # --- Stability filtering ---
        stability_n_splits=3,
        stability_min_profitable_splits=2,
        # --- Follower filtering (drawdown-based) ---
        follower_max_dd_pnl_ratio=0.3,
        # --- Pair filtering ---
        min_pair_observations=3,
        # --- Time window ---
        time_window_minutes=30,
    )
else:
    raise ValueError(f"Unsupported DEFAULT_TAGS: {DEFAULT_TAGS}")

for k, v in PARAMS.items():
    print(f"  {k}: {v}")

  min_follower_copyable_roi: 0.3
  min_follower_trade_value: 100
  min_follower_num_buckets: 30
  max_follower_hhi: 0.3
  min_leader_trade_count: 20
  max_leader_hhi: 1
  stability_min_follower_roi: 0.0
  stability_min_follower_buckets: 10
  stability_max_follower_hhi: 1
  max_dd_pnl_ratio: 0.3
  follower_max_dd_pnl_ratio: 0.3
  min_pair_observations: 3
  time_window_minutes: 15


## Load data

In [3]:
df_full = load_trades()
df_full = compute_copyable_notional(df_full)

train_cutoff = pd.Timestamp("2026-06-01", tz="UTC")
val_cutoff = pd.Timestamp("2026-07-01", tz="UTC")

df_train = df_full[df_full["dt"] < train_cutoff].copy()
df_val = df_full[(df_full["dt"] >= train_cutoff) & (df_full["dt"] < val_cutoff)].copy()
df_test = df_full[df_full["dt"] >= val_cutoff].copy()

print(f"Split by trade date:")
print(f"  Train: {len(df_train):>10,} trades  ({df_train['condition_id'].nunique():>5,} markets)  < {train_cutoff.date()}")
print(f"  Val:   {len(df_val):>10,} trades  ({df_val['condition_id'].nunique():>5,} markets)  {train_cutoff.date()} .. {val_cutoff.date()}")
print(f"  Test:  {len(df_test):>10,} trades  ({df_test['condition_id'].nunique():>5,} markets)  >= {val_cutoff.date()}")
print(f"  Total: {len(df_full):>10,} trades  ({df_full['condition_id'].nunique():>5,} markets)")

# Market overlap check
train_markets = set(df_train["condition_id"].unique())
val_markets = set(df_val["condition_id"].unique())
test_markets = set(df_test["condition_id"].unique())
print(f"\n  Markets overlapping train/val: {len(train_markets & val_markets)}")
print(f"  Markets overlapping train/test: {len(train_markets & test_markets)}")
print(f"  Markets overlapping val/test: {len(val_markets & test_markets)}")

Markets: 1974837


Filtered markets for {'Weather'}: 91123
Loading 16 trade shards...


Total trades loaded: 14,250,603


Unique wallets: 4,082
Date range: 2025-01-09 15:32:39+00:00 -> 2026-07-27 06:12:25+00:00


Split by trade date:


  Train:  6,035,492 trades  (23,237 markets)  < 2026-06-01


  Val:    4,766,255 trades  (19,787 markets)  2026-06-01 .. 2026-07-01
  Test:   3,448,856 trades  (16,333 markets)  >= 2026-07-01


  Total: 14,250,603 trades  (56,875 markets)



  Markets overlapping train/val: 1350
  Markets overlapping train/test: 1
  Markets overlapping val/test: 1132


## Compute wallet metrics on training data

In [4]:
wallet_vol, _ = compute_wallet_metrics(df_train)

wallet_vol["copyable_pnl_factor"] = np.clip(
    wallet_vol["copyable_pnl"] / wallet_vol["total_pnl"].replace(0, np.nan),
    0, 1.0,
).fillna(0.0)
wallet_vol["copyable_roi"] = wallet_vol["average_roi"] * wallet_vol["copyable_pnl_factor"]

opening_metrics = compute_opening_metrics(df_train)
wallet_vol = wallet_vol.merge(opening_metrics, on="wallet", how="left")
for c in ["opening_roi", "opening_pnl", "opening_copyable_roi", "opening_copyable_pnl"]:
    wallet_vol[c] = wallet_vol[c].fillna(0.0)

print(f"Wallets with metrics: {len(wallet_vol)}")
wallet_vol[["wallet", "buy_roi", "sell_roi", "copyable_pnl", "copyable_roi", "num_buckets"]].head(10)

Wallets with metrics: 3584


,wallet,buy_roi,sell_roi,copyable_pnl,copyable_roi,num_buckets
0,0x00546dfeb4097e232c86a775ccbe3c9b84c0cab1,0.0236,NaN,12.8458,0.0186,45
1,0x0054ee7dfb882d2d016fa13ef5f5cdb3b0ebcf1f,0.0051,0.0020,-3.3317,0.0000,212
2,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0.0262,0.0360,227.3151,-0.0020,5300
3,0x00833cc2d777e6f2fc8437679124024ae6468cb1,NaN,NaN,0.0000,NaN,2
4,0x0141be702d272f17666e280303ad44e7bc0cc2da,0.6776,-1.0000,119.3868,0.5030,34
5,0x015be8bad14c79d2722a0bd8bbe0cd93b905556d,NaN,NaN,-20.4954,NaN,299
6,0x01a5fb1fa13f378138a31382c8364b5d4e2b0e36,0.0120,-1.0000,12.6813,-0.0480,46
7,0x01a68281185e728ba0fef6245008bf8af68a59b0,0.0259,-0.1815,136.2436,0.0063,4360
8,0x01ced860d8dca5d7987579d2a2635df8520d27a2,0.1968,-0.0635,-316.5350,0.0000,1356
9,0x01d94480e2a96cdd01fed071878b1adf82e0acd0,0.0058,-0.9697,24.2417,0.0176,2454


## Baseline selection

In [5]:
follower_wallets = select_follower_wallets(
    wallet_vol,
    min_copyable_roi=PARAMS["min_follower_copyable_roi"],
    min_trade_value=PARAMS["min_follower_trade_value"],
    min_num_buckets=PARAMS["min_follower_num_buckets"],
    max_market_pnl_hhi=PARAMS["max_follower_hhi"],
)
print(f"Followers: {len(follower_wallets)}")

buy_leaders = select_leader_wallets(
    wallet_vol,
    min_trade_count=PARAMS["min_leader_trade_count"],
    min_roi=None,
    max_market_pnl_hhi=PARAMS["max_leader_hhi"],
    side="BUY",
)
print(f"Buy leaders: {len(buy_leaders)}")

sell_leaders = select_leader_wallets(
    wallet_vol,
    min_trade_count=PARAMS["min_leader_trade_count"],
    min_roi=None,
    max_market_pnl_hhi=PARAMS["max_leader_hhi"],
    side="SELL",
)
print(f"Sell leaders: {len(sell_leaders)}")

Followers: 44
Buy leaders: 1401
Sell leaders: 1401


## Baseline evaluation

In [6]:
follower_ws = set(follower_wallets['wallet'])
buy_leader_ws = set(buy_leaders['wallet'])
sell_leader_ws = set(sell_leaders['wallet'])
tw = PARAMS["time_window_minutes"]

for split_name, df_split in  [("TRAIN", df_train), ("VAL", df_val), ("TEST", df_test)]:
    buy_ev = evaluate_implied_pnl(
        df_split, follower_ws, buy_leader_ws,
        time_window_minutes=tw, leader_side="BUY"
    )
    sell_ev = evaluate_implied_pnl(
        df_split, follower_ws, sell_leader_ws,
        time_window_minutes=tw, leader_side="SELL",
    )
    total = buy_ev["followed_copyable_pnl"] + sell_ev["followed_copyable_pnl"]
    notional = buy_ev["followed_copyable_notional"] + sell_ev["followed_copyable_notional"]
    roi = total / notional if notional > 0 else 0
    print(f"{split_name}: buy_pnl={buy_ev['followed_copyable_pnl']:.2f} ({buy_ev['trade_count']} trades, {buy_ev['leader_count']} leaders)  "
          f"sell_pnl={sell_ev['followed_copyable_pnl']:.2f} ({sell_ev['trade_count']} trades, {sell_ev['leader_count']} leaders)  "
          f"total={total:.2f}  roi={roi:.4f}")

TRAIN: buy_pnl=27774.11 (11958 trades, 744 leaders)  sell_pnl=29475.75 (13978 trades, 550 leaders)  total=57249.86  roi=0.5494


VAL: buy_pnl=1316.33 (11277 trades, 540 leaders)  sell_pnl=1635.48 (14589 trades, 423 leaders)  total=2951.81  roi=0.0206


TEST: buy_pnl=5564.15 (9373 trades, 429 leaders)  sell_pnl=7774.24 (12879 trades, 298 leaders)  total=13338.40  roi=0.1359


## Score leaders (baseline)

In [7]:
# Buy leaders
buy_implied = detect_implied_buys(
    df_train, follower_ws, buy_leader_ws,
    time_window_minutes=tw, leader_side="BUY",
)
buy_scores = score_leaders(buy_implied)
print("Top buy leaders:")
print(buy_scores.head(10).to_string())

print()

# Sell leaders
sell_implied = detect_implied_buys(
    df_train, follower_ws, sell_leader_ws,
    time_window_minutes=tw, leader_side="SELL",
)
sell_scores = score_leaders(sell_implied)
print("Top sell leaders:")
print(sell_scores.head(10).to_string())

Top buy leaders:
                                leader_wallet  num_followers  total_follower_copyable_pnl  num_followed_trades  unique_tokens
0  0x3dc44175ae2d0175ee7bc76cd9ec04b614a91fd8              9                    4009.9218                   31             23
1  0xde0fa43faad1c0da4881c63681ad07a8c850a4e7              1                    2227.6980                    1              1
2  0x48180cfe026031c280ebdea2acad996867cde5de             16                    1960.9431                   78             52
3  0x57ee70867b4e387de9de34fd62bc685aa02a8112              9                    1838.1337                   19             17
4  0x50b977391c4b3dd88b0a0bef03c3434fe4284298             13                    1538.1732                   60             54
5  0x26123cbf0f4820f7e70408a8c054ba7615c05289             34                    1158.6849                  892            515
6  0x68375b8cd026ad970a748c3f385c15e4772e1108              3                    1103.9888            

Top sell leaders:
                                leader_wallet  num_followers  total_follower_copyable_pnl  num_followed_trades  unique_tokens
0  0x177500541ae20bb0d46ab0db3fd2559e2a7e85b0              3                    9330.0049                   12              3
1  0xb40e89677d59665d5188541ad860450a6e2a7cc9             41                    5827.2352                 1271            593
2  0x48180cfe026031c280ebdea2acad996867cde5de             17                    2548.0287                  194             56
3  0x26123cbf0f4820f7e70408a8c054ba7615c05289             38                    1912.9634                 1778            534
4  0x77bdeb3f229bf6d826d20dbd5f0c7972c32ae48f             26                    1324.6071                  635            175
5  0x01f2a8baabe17c2541d1e3091220991f257ac3de             34                    1131.8465                  237            126
6  0x945a49252f772a10c6ddd1d1e1e24ee20438a48c             39                    1085.7841           

## Improved pipeline: Stable leaders + followers + pair filtering

Filter leaders who are consistently profitable across training time slices,
filter followers whose implied PnL has low drawdown, then require a minimum
number of observed copy-trades per leader-follower pair.

In [8]:
# Broad follower set for leader stability detection (more implied trades per leader)
fw_broad = set(select_follower_wallets(
    wallet_vol,
    min_copyable_roi=PARAMS["stability_min_follower_roi"],
    min_trade_value=PARAMS["min_follower_trade_value"],
    min_num_buckets=PARAMS["stability_min_follower_buckets"],
    max_market_pnl_hhi=PARAMS["stability_max_follower_hhi"],
)["wallet"])

if DEFAULT_TAGS == {"Weather"}:
    stable_buy_leaders = filter_leaders_by_drawdown(
        df_train, buy_leader_ws, fw_broad,
        time_window_minutes=tw, leader_side="BUY",
        max_dd_pnl_ratio=PARAMS["max_dd_pnl_ratio"],
    )
    stable_sell_leaders = filter_leaders_by_drawdown(
        df_train, sell_leader_ws, fw_broad,
        time_window_minutes=tw, leader_side="SELL",
        max_dd_pnl_ratio=PARAMS["max_dd_pnl_ratio"],
    )
elif DEFAULT_TAGS == {"Politics"}:
    stable_buy_leaders = filter_stable_leaders(
        df_train, buy_leader_ws, fw_broad,
        time_window_minutes=tw, leader_side="BUY",
        n_splits=PARAMS["stability_n_splits"],
        min_profitable_splits=PARAMS["stability_min_profitable_splits"],
    )
    stable_sell_leaders = filter_stable_leaders(
        df_train, sell_leader_ws, fw_broad,
        time_window_minutes=tw, leader_side="SELL",
        n_splits=PARAMS["stability_n_splits"],
        min_profitable_splits=PARAMS["stability_min_profitable_splits"],
    )
else:
    raise ValueError(f"Unsupported DEFAULT_TAGS: {DEFAULT_TAGS}")

print(f"Stable buy leaders: {len(stable_buy_leaders)} / {len(buy_leader_ws)}")
print(f"Stable sell leaders: {len(stable_sell_leaders)} / {len(sell_leader_ws)}")

buy_imp_train = detect_implied_buys(
    df_train, follower_ws, stable_buy_leaders,
    time_window_minutes=tw, leader_side="BUY",
)
sell_imp_train = detect_implied_buys(
    df_train, follower_ws, stable_sell_leaders,
    time_window_minutes=tw, leader_side="SELL",
)

min_obs = PARAMS["min_pair_observations"]
buy_pairs = filter_pairs_by_frequency(buy_imp_train, min_observations=min_obs)
sell_pairs = filter_pairs_by_frequency(sell_imp_train, min_observations=min_obs)

final_buy_leaders = set(buy_pairs["leader_wallet"]) if not buy_pairs.empty else set()
final_sell_leaders = set(sell_pairs["leader_wallet"]) if not sell_pairs.empty else set()
final_followers = (
    (set(buy_pairs["follower_wallet"]) if not buy_pairs.empty else set())
    | (set(sell_pairs["follower_wallet"]) if not sell_pairs.empty else set())
)

# Filter followers by drawdown on TRAIN+VAL combined implied trades
df_train_val = pd.concat([df_train, df_val], ignore_index=True)
buy_imp_train_val = detect_implied_buys(
    df_train_val, follower_ws, stable_buy_leaders,
    time_window_minutes=tw, leader_side="BUY",
)
sell_imp_train_val = detect_implied_buys(
    df_train_val, follower_ws, stable_sell_leaders,
    time_window_minutes=tw, leader_side="SELL",
)
follower_dd_ratio = PARAMS.get("follower_max_dd_pnl_ratio", 0.3)
stable_followers = filter_followers_by_drawdown(
    buy_imp_train_val, sell_imp_train_val,
    max_dd_pnl_ratio=follower_dd_ratio,
)
removed_by_dd = final_followers - stable_followers
final_followers = final_followers & stable_followers

# Filter followers by VAL ROI (generalization check)
min_val_roi = 0.07
val_roi_followers = filter_followers_by_val_roi(
    df_val, final_followers, final_buy_leaders, final_sell_leaders,
    time_window_minutes=tw, min_val_roi=min_val_roi,
)
removed_by_val = final_followers - val_roi_followers
final_followers = final_followers & val_roi_followers

print()  # newline before summary
print(f"After pair filter (min_obs={min_obs}):")
print(f"  Followers: {len(final_followers)} (dd removed {len(removed_by_dd)}, val_roi removed {len(removed_by_val)})")
print(f"  Buy leaders: {len(final_buy_leaders)}")
print(f"  Sell leaders: {len(final_sell_leaders)}")

Stable buy leaders: 529 / 1401
Stable sell leaders: 415 / 1401



After pair filter (min_obs=3):
  Followers: 6 (dd removed 19, val_roi removed 18)
  Buy leaders: 132
  Sell leaders: 77


In [9]:
print("=" * 70)
print(f"EVALUATION (tw={tw}min, min_pair_obs={min_obs})")
print("=" * 70)

for split_name, df_split in [("TRAIN", df_train), ("VAL", df_val), ("TEST", df_test)]:
    buy_ev = evaluate_implied_pnl(df_split, final_followers, final_buy_leaders, time_window_minutes=tw, leader_side="BUY")
    sell_ev = evaluate_implied_pnl(df_split, final_followers, final_sell_leaders, time_window_minutes=tw, leader_side="SELL")
    follower_buy = evaluate_follower_buy_performance(df_split, final_followers)

    n_active = len(set(df_split[df_split["wallet"].isin(final_followers)]["wallet"]))
    n_markets = df_split["condition_id"].nunique()

    imp_pnl = buy_ev["followed_copyable_pnl"] + sell_ev["followed_copyable_pnl"]
    imp_notional = buy_ev["followed_copyable_notional"] + sell_ev["followed_copyable_notional"]
    imp_trades = buy_ev["trade_count"] + sell_ev["trade_count"]
    imp_roi = imp_pnl / imp_notional if imp_notional > 0 else 0.0

    b_roi = buy_ev["followed_copyable_pnl"] / buy_ev["followed_copyable_notional"] if buy_ev["followed_copyable_notional"] > 0 else 0.0
    s_roi = sell_ev["followed_copyable_pnl"] / sell_ev["followed_copyable_notional"] if sell_ev["followed_copyable_notional"] > 0 else 0.0

    print(f"\n{split_name} ({n_markets} markets, {n_active} active followers):")
    print(f"  Implied BUY:   Copyable PnL: {buy_ev['followed_copyable_pnl']:>10.2f}  ROI: {b_roi:>7.4f}  ({buy_ev['trade_count']} trades, {buy_ev['leader_count']} leaders)")
    print(f"  Implied SELL:  Copyable PnL: {sell_ev['followed_copyable_pnl']:>10.2f}  ROI: {s_roi:>7.4f}  ({sell_ev['trade_count']} trades, {sell_ev['leader_count']} leaders)")
    print(f"  Implied total: Copyable PnL: {imp_pnl:>10.2f}  ROI: {imp_roi:>7.4f}  ({imp_trades} trades)")
    print(f"  All buys:      Copyable PnL: {follower_buy['followed_copyable_pnl']:>10.2f}  ROI: {follower_buy['followed_copyable_roi']:>7.4f}  (wallet PnL: {follower_buy['wallet_pnl']:>10.2f}, {follower_buy['trade_count']} trades)")

EVALUATION (tw=15min, min_pair_obs=3)



TRAIN (23237 markets, 6 active followers):
  Implied BUY:   Copyable PnL:    1993.77  ROI:  0.3285  (772 trades, 77 leaders)
  Implied SELL:  Copyable PnL:    2402.37  ROI:  0.3476  (996 trades, 46 leaders)
  Implied total: Copyable PnL:    4396.13  ROI:  0.3387  (1768 trades)
  All buys:      Copyable PnL:    2879.98  ROI:  0.3181  (wallet PnL:    8681.37, 1568 trades)



VAL (19787 markets, 6 active followers):
  Implied BUY:   Copyable PnL:    2576.90  ROI:  0.1494  (1114 trades, 71 leaders)
  Implied SELL:  Copyable PnL:    3021.82  ROI:  0.1585  (1298 trades, 34 leaders)
  Implied total: Copyable PnL:    5598.72  ROI:  0.1542  (2412 trades)
  All buys:      Copyable PnL:    3588.62  ROI:  0.0947  (wallet PnL:   28945.70, 2276 trades)



TEST (16333 markets, 5 active followers):
  Implied BUY:   Copyable PnL:    2299.04  ROI:  0.1617  (1425 trades, 46 leaders)
  Implied SELL:  Copyable PnL:    2511.78  ROI:  0.1540  (1611 trades, 30 leaders)
  Implied total: Copyable PnL:    4810.82  ROI:  0.1576  (3036 trades)
  All buys:      Copyable PnL:    3507.85  ROI:  0.1426  (wallet PnL:   30415.84, 2770 trades)


## Summary

In [10]:
# Store for save cell
b_fw = final_followers
b_blw = final_buy_leaders
b_slw = final_sell_leaders

print(f"Final wallet counts:")
print(f"  Followers: {len(b_fw)}")
print(f"  Buy leaders: {len(b_blw)}")
print(f"  Sell leaders: {len(b_slw)}")

Final wallet counts:
  Followers: 6
  Buy leaders: 132
  Sell leaders: 77


In [11]:
best_params = PARAMS.copy()
print("Pipeline params:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

Pipeline params:
  min_follower_copyable_roi: 0.3
  min_follower_trade_value: 100
  min_follower_num_buckets: 30
  max_follower_hhi: 0.3
  min_leader_trade_count: 20
  max_leader_hhi: 1
  stability_min_follower_roi: 0.0
  stability_min_follower_buckets: 10
  stability_max_follower_hhi: 1
  max_dd_pnl_ratio: 0.3
  follower_max_dd_pnl_ratio: 0.3
  min_pair_observations: 3
  time_window_minutes: 15


## Concentration diagnostics (test split)

In [12]:
buy_imp_test = detect_implied_buys(df_test, b_fw, b_blw, time_window_minutes=tw, leader_side="BUY")
sell_imp_test = detect_implied_buys(df_test, b_fw, b_slw, time_window_minutes=tw, leader_side="SELL")
imp_test = pd.concat([buy_imp_test, sell_imp_test], ignore_index=True)

if imp_test.empty:
    print("No implied trades on test set.")
else:
    total_pnl = imp_test["copyable_pnl"].sum()
    total_trades = len(imp_test)
    print(f"Test implied trades: {total_trades:,}   Total copyable PnL: ${total_pnl:,.2f}\n")

    # --- Leader concentration ---
    leader_pnl = (
        imp_test.groupby("leader_wallet", sort=False)["copyable_pnl"]
        .agg(["sum", "count", "nunique"])
        .rename(columns={"sum": "pnl", "count": "trades", "nunique": "followers"})
        .sort_values("pnl", ascending=False)
    )
    leader_pnl["cum_pnl"] = leader_pnl["pnl"].cumsum()
    leader_pnl["cum_pct"] = leader_pnl["cum_pnl"] / total_pnl
    n_leaders = len(leader_pnl)
    print(f"Leaders contributing to test PnL: {n_leaders}")
    for k in [1, 3, 5, 10]:
        if k <= n_leaders:
            pct = leader_pnl.iloc[k - 1]["cum_pct"]
            print(f"  Top {k:>2} leader(s): ${leader_pnl.iloc[k - 1]['cum_pnl']:>10,.2f}  ({pct:.1%} of total)")
    print()
    print("Top 10 leaders:")
    print(leader_pnl.head(10).to_string())

    # --- Follower concentration ---
    follower_pnl = (
        imp_test.groupby("follower_wallet", sort=False)["copyable_pnl"]
        .agg(["sum", "count"])
        .rename(columns={"sum": "pnl", "count": "trades"})
        .sort_values("pnl", ascending=False)
    )
    follower_pnl["cum_pnl"] = follower_pnl["pnl"].cumsum()
    follower_pnl["cum_pct"] = follower_pnl["cum_pnl"] / total_pnl
    n_followers = len(follower_pnl)
    print(f"\nFollowers active on test: {n_followers}")
    for k in [1, 5, 10, 20]:
        if k <= n_followers:
            pct = follower_pnl.iloc[k - 1]["cum_pct"]
            print(f"  Top {k:>2} follower(s): ${follower_pnl.iloc[k - 1]['cum_pnl']:>10,.2f}  ({pct:.1%} of total)")
    print()
    print("Top 10 followers:")
    print(follower_pnl.head(10).to_string())

    # --- Market concentration ---
    market_pnl = (
        imp_test.groupby("condition_id", sort=False)["copyable_pnl"]
        .agg(["sum", "count"])
        .rename(columns={"sum": "pnl", "count": "trades"})
        .sort_values("pnl", ascending=False)
    )
    market_pnl["cum_pnl"] = market_pnl["pnl"].cumsum()
    market_pnl["cum_pct"] = market_pnl["cum_pnl"] / total_pnl
    n_markets = len(market_pnl)
    print(f"\nMarkets with implied trades: {n_markets}")
    for k in [1, 3, 5, 10]:
        if k <= n_markets:
            pct = market_pnl.iloc[k - 1]["cum_pct"]
            print(f"  Top {k:>2} market(s):  ${market_pnl.iloc[k - 1]['cum_pnl']:>10,.2f}  ({pct:.1%} of total)")
    print()
    print("Top 10 markets:")
    print(market_pnl.head(10).to_string())

    # --- Negative PnL followers ---
    neg_followers = (follower_pnl["pnl"] < 0).sum()
    neg_pnl = follower_pnl.loc[follower_pnl["pnl"] < 0, "pnl"].sum()
    print(f"\nFollowers with negative PnL: {neg_followers}  (total: ${neg_pnl:,.2f})")
    pos_followers = (follower_pnl["pnl"] > 0).sum()
    pos_pnl = follower_pnl.loc[follower_pnl["pnl"] > 0, "pnl"].sum()
    print(f"Followers with positive PnL: {pos_followers}  (total: ${pos_pnl:,.2f})")

    # --- Gini coefficient on leader PnL ---
    vals = leader_pnl["pnl"].values
    vals_sorted = np.sort(vals)
    n = len(vals_sorted)
    cum = np.cumsum(vals_sorted)
    gini = 1 - 2 * np.sum(cum) / (n * cum[-1]) if cum[-1] > 0 else 0.0
    print(f"\nLeader PnL Gini coefficient: {gini:.4f}  (1 = perfect concentration, 0 = equal)")

Test implied trades: 3,036   Total copyable PnL: $4,810.82

Leaders contributing to test PnL: 67
  Top  1 leader(s): $  1,293.54  (26.9% of total)
  Top  3 leader(s): $  3,115.50  (64.8% of total)
  Top  5 leader(s): $  3,669.07  (76.3% of total)
  Top 10 leader(s): $  4,444.66  (92.4% of total)

Top 10 leaders:
                                                 pnl  trades  followers   cum_pnl  cum_pct
leader_wallet                                                                             
0xb40e89677d59665d5188541ad860450a6e2a7cc9 1293.5422     591        376 1293.5422   0.2689
0x945a49252f772a10c6ddd1d1e1e24ee20438a48c 1168.1621     942        540 2461.7043   0.5117
0x26123cbf0f4820f7e70408a8c054ba7615c05289  653.7991     153        109 3115.5034   0.6476
0x1abc2b469b5ded495d6c8b21fc79dcb8d8f345e7  292.3331      53         29 3407.8364   0.7084
0x0335cb69dd7498a24c39edea33c925ce51d36eca  261.2355      18         10 3669.0720   0.7627
0x510f4963b66b1b18505faab74b0bb943d1dda43c  227.8

## Save stage 1 result

In [13]:
import json
from datetime import datetime, timezone
from pathlib import Path


def _convert(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


# Collect wallet records for each group
wallet_cols = [
    "wallet", "buy_roi", "sell_roi", "copyable_pnl", "copyable_roi",
    "num_buckets", "num_markets", "total_notional", "total_pnl",
    "market_pnl_hhi",
]

def _wallet_records(df):
    if df is None or df.empty:
        return []
    cols = [c for c in wallet_cols if c in df.columns]
    records = df[cols].to_dict(orient="records")
    return [{k: _convert(v) for k, v in w.items()} for w in records]


metadata = {
    "type": "implied",
    "tags": sorted(DEFAULT_TAGS),
    "run_timestamp": datetime.now(timezone.utc).isoformat(),
    "n_followers": len(b_fw),
    "n_buy_leaders": len(b_blw),
    "n_sell_leaders": len(b_slw),
    "n_wallets_total": len(wallet_vol),
}

payload = {
    "stage": 1,
    "type": "implied",
    "best_params": {k: _convert(v) for k, v in best_params.items()},
    "metadata": metadata,
    "wallets": {
        "followers": _wallet_records(follower_wallets[follower_wallets["wallet"].isin(b_fw)]),
        "buy_leaders": _wallet_records(buy_leaders[buy_leaders["wallet"].isin(b_blw)]),
        "sell_leaders": _wallet_records(sell_leaders[sell_leaders["wallet"].isin(b_slw)]),
    },
}

out_path = Path("stage1_implied_result.json")
with open(out_path, "w") as f:
    json.dump(payload, f, indent=2)
print(f"Saved stage 1 implied result -> {out_path.resolve()}")

Saved stage 1 implied result -> /Users/vobornij/projects/polymarket/notebooks/wallet_selection/stage1_implied_result.json


In [14]:
df_test.columns

Index(['wallet', 'condition_id', 'token_id', 'dt', 'side', 'position',
       'quantity', 'price', 'usdc_amount', 'final_value_usdc', 'trade_pnl',
       'copyable_pnl', 'token_winner', 'final_price',
       'last_condition_trade_ts', 'tx_hash', 'num_fills', 'is_train',
       'copyable_qty', 'avail_copy_total_vol', 'avail_copy_count',
       'end_date_iso', 'question', 'tags', 'primary_tag', 'winner_token_id',
       'outcome', 'pnl', 'notional', 'copyable_notional', 'roi',
       'copyable_roi'],
      dtype='str')

In [15]:
df_test[
    (df_test['condition_id'] == '0x18d19ee1593c5f758c9b652fae58b0ea98e8b28f3cc6c6ce42816dd032f9c8a7')
    & (df_test['dt'] >= pd.Timestamp("2026-07-09 07:37:31+00:00", tz="UTC"))
    & (df_test['pnl'] >= 0)
    & (df_test['dt'] <= pd.Timestamp("2026-07-09 07:42:31+00:00", tz="UTC"))
    # & (df_test['price'] <= 0.149)
       ].sort_values("dt")[['dt', 'wallet', 'side', 'price', 'trade_pnl', 'copyable_pnl', 'token_id', 'tx_hash']]

,dt,wallet,side,price,trade_pnl,copyable_pnl,token_id,tx_hash
1352015,2026-07-09 07:37:31+00:00,0x945a49252f772a10c6ddd1d1e1e24ee20438a48c,SELL,0.9490,2.7726,0.0000,2254165225066728302189294187670622099851703025...,0x47068d48c7375880cf06fb2eed3ec891e1fa31922caa...
1352016,2026-07-09 07:37:31+00:00,0x945a49252f772a10c6ddd1d1e1e24ee20438a48c,BUY,0.0510,1.2174,0.0000,3120308255357225723884420077341617187891774522...,0x47068d48c7375880cf06fb2eed3ec891e1fa31922caa...
1411723,2026-07-09 07:37:31+00:00,0xafde461fce5aa0fabdb7711c59db93b65e343e1d,SELL,0.8656,164.3030,163.4001,2254165225066728302189294187670622099851703025...,0x4f43e1da2143ef966eb26aa4bf77c759d42a1dfc5a70...
1411724,2026-07-09 07:37:31+00:00,0xafde461fce5aa0fabdb7711c59db93b65e343e1d,BUY,0.1496,600.5540,187.7898,3120308255357225723884420077341617187891774522...,0x4f43e1da2143ef966eb26aa4bf77c759d42a1dfc5a70...
908618,2026-07-09 07:37:38+00:00,0x0a8eee5cb1f039540abbbecc03571146048c05a4,BUY,0.1600,72.9708,72.9708,3120308255357225723884420077341617187891774522...,0x19f2f7c81255c35a80a79731b28576c19e10798973bf...
1292515,2026-07-09 07:37:38+00:00,0x8f514910c5e1f2ce541d01d8bbd8e9bea9638b09,BUY,0.1967,20.3806,20.3806,3120308255357225723884420077341617187891774522...,0x0bb56bf4f13cc4d1318f2841a7631cef4fe647560698...
1411725,2026-07-09 07:37:52+00:00,0xafde461fce5aa0fabdb7711c59db93b65e343e1d,BUY,0.1505,88.2706,87.7934,3120308255357225723884420077341617187891774522...,0xf0dac8e4bcb002f490bc07246233116dc6a39b15d440...


In [16]:
buy_imp_test.groupby('condition_id').agg(
    num_trades=('copyable_pnl', 'count'),
    total_copyable_pnl=('copyable_pnl', 'sum'),
    total_copyable_notional=('copyable_notional', 'sum')
).sort_values(by='total_copyable_pnl', ascending=False).head(10)

,num_trades,total_copyable_pnl,total_copyable_notional
condition_id,,,
0xf2c9a5f2c7adf9cd8d7605d7616f5c5dbc3a048d8f5416a479804cd7006d648c,18,727.9148,485.8059
0xd66bef9f47a75db344f0014d43770db425f4bfa4a06ab678b41f8efeafc7c7e2,28,421.5707,681.2340
0xdc6ae20b7f34fa08a4dbf2b6f7da3b9fe0ba42adb7ee5eecee38ccd900eeb77d,21,265.5424,184.0209
0x8a204142c8eccaf5679a7e5f63c22347542778ff50a859ff270d81090e57e469,1,261.2355,745.3928
0xefb267bab516e5039ba2e1763aa2f3d3788cdfb1900418077068f52008149d1d,7,186.6722,208.6441
0x1cd9659d120bcdc4c570d1282513c909fa98f0cb72298f7638a3e7596d9f867a,8,166.0024,157.7676
0x16341b2dea5d30d8c337ad7d3835b6e6bc4a61015deb272f1fa82c1ba7459460,25,155.9226,274.5255
0x7c47a4e04e88f42aa3d53fa56df2089bac4babcc811ad95c82f80a8c130c64c8,25,146.1280,1049.0619
0x90eb0a2be6365080fc7bae77972df656440044808a383a6ab1298d196d51ec74,17,141.3174,298.6274


In [17]:
(
    buy_imp_test[buy_imp_test['condition_id'] == '0x18d19ee1593c5f758c9b652fae58b0ea98e8b28f3cc6c6ce42816dd032f9c8a7']
)

,follower_wallet,condition_id,outcome,follower_dt,pnl,notional,copyable_pnl,copyable_notional,leader_wallet,leader_dt,time_delta_seconds
